<a href="https://colab.research.google.com/github/ImalYH/Football-Power-Ranking-System-w1952877/blob/main/InterimCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
#install required packages
!pip install catboost gdown --quiet

#import libraries
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import gdown


In [40]:
#install required libraries if missing
def install_package(package):
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)

install_package("catboost")
install_package("gdown")
install_package("pandas")
install_package("numpy")
install_package("scikit-learn")

Installing scikit-learn...


In [41]:
#download dataset if not available
dataset_id = "1DhGT608tKIybKUOSeU-zjnP9CZ7ykf4S"
dataset_filename = "PL_Combined_2016_2024.csv"

if not os.path.exists(dataset_filename):
    gdown.download(id=dataset_id, output=dataset_filename, quiet=False)

In [42]:
#load dataset
df = pd.read_csv(dataset_filename)
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")


In [43]:
#feature engineering

#recent form function
def calculate_recent_form(team, date, column):
    recent_matches = df[(df["HomeTeam"] == team) | (df["AwayTeam"] == team)]
    recent_matches = recent_matches[recent_matches["Date"] < date].sort_values(by="Date", ascending=False).head(5)

    if recent_matches.empty:
        return 0

    if column == "Goals Scored":
        return recent_matches["FTHG"].where(recent_matches["HomeTeam"] == team, recent_matches["FTAG"]).mean()
    elif column == "Goals Conceded":
        return recent_matches["FTAG"].where(recent_matches["HomeTeam"] == team, recent_matches["FTHG"]).mean()

df["Home_RecentGoals"] = df.apply(lambda row: calculate_recent_form(row["HomeTeam"], row["Date"], "Goals Scored"), axis=1)
df["Away_RecentGoals"] = df.apply(lambda row: calculate_recent_form(row["AwayTeam"], row["Date"], "Goals Scored"), axis=1)

In [44]:
#home advantage score
home_avg_goals = df.groupby("HomeTeam")["FTHG"].mean()
away_avg_goals = df.groupby("AwayTeam")["FTAG"].mean()
df["Home_Advantage"] = df["HomeTeam"].map(home_avg_goals) - df["AwayTeam"].map(away_avg_goals)

In [45]:
#head to head performance
def calculate_head_to_head(home_team, away_team):
    past_matches = df[((df["HomeTeam"] == home_team) & (df["AwayTeam"] == away_team)) |
                      ((df["HomeTeam"] == away_team) & (df["AwayTeam"] == home_team))]
    past_matches = past_matches.sort_values(by="Date", ascending=False).head(5)

    if past_matches.empty:
        return 0

    return past_matches["FTHG"].mean() - past_matches["FTAG"].mean()

df["HeadToHead"] = df.apply(lambda row: calculate_head_to_head(row["HomeTeam"], row["AwayTeam"]), axis=1)

In [47]:
#prepare training data
features = ["HomeTeam", "AwayTeam", "Referee", "Home_RecentGoals", "Away_RecentGoals", "Home_Advantage", "HeadToHead"]
target_home = "FTHG"
target_away = "FTAG"

df[["HomeTeam", "AwayTeam", "Referee"]] = df[["HomeTeam", "AwayTeam", "Referee"]].astype("category")

In [48]:
#train test split
x = df[features]
y_home = df[target_home]
y_away = df[target_away]

x_train, x_test, y_train_home, y_test_home = train_test_split(x, y_home, test_size=0.2, random_state=42)
_, _, y_train_away, y_test_away = train_test_split(x, y_away, test_size=0.2, random_state=42)

In [49]:
#train catBoost models
catboost_params = {
    "iterations": 500,
    "depth": 6,
    "learning_rate": 0.1,
    "loss_function": "MAE",
    "cat_features": ["HomeTeam", "AwayTeam", "Referee"],
    "verbose": 100
}

model_home = CatBoostRegressor(**catboost_params)
model_home.fit(x_train, y_train_home)

model_away = CatBoostRegressor(**catboost_params)
model_away.fit(x_train, y_train_away)

0:	learn: 1.0198182	total: 5.5ms	remaining: 2.75s
100:	learn: 0.8110489	total: 326ms	remaining: 1.29s
200:	learn: 0.7303985	total: 680ms	remaining: 1.01s
300:	learn: 0.6795739	total: 1.01s	remaining: 666ms
400:	learn: 0.6414889	total: 1.35s	remaining: 333ms
499:	learn: 0.6110147	total: 1.71s	remaining: 0us
0:	learn: 0.8904186	total: 4.64ms	remaining: 2.31s
100:	learn: 0.7568088	total: 321ms	remaining: 1.27s
200:	learn: 0.6863880	total: 650ms	remaining: 967ms
300:	learn: 0.6335665	total: 999ms	remaining: 660ms
400:	learn: 0.6028298	total: 1.32s	remaining: 327ms
499:	learn: 0.5721496	total: 1.66s	remaining: 0us


In [50]:
#define prediction function
def predict_match(home_team, away_team, referee):
    input_data = pd.DataFrame({
        "HomeTeam": [home_team],
        "AwayTeam": [away_team],
        "Referee": [referee],
        "Home_RecentGoals": [calculate_recent_form(home_team, pd.to_datetime("2024-05-01"), "Goals Scored")],
        "Away_RecentGoals": [calculate_recent_form(away_team, pd.to_datetime("2024-05-01"), "Goals Scored")],
        "Home_Advantage": [home_avg_goals.get(home_team, 0) - away_avg_goals.get(away_team, 0)],
        "HeadToHead": [calculate_head_to_head(home_team, away_team)]
    })

    home_goals_pred = model_home.predict(input_data)[0]
    away_goals_pred = model_away.predict(input_data)[0]

    return round(home_goals_pred), round(away_goals_pred)

In [51]:
#allow user input
try:
    from ipywidgets import widgets
    from IPython.display import display

    home_team_widget = widgets.Text(placeholder="Enter Home Team")
    away_team_widget = widgets.Text(placeholder="Enter Away Team")
    referee_widget = widgets.Text(placeholder="Enter Referee")
    button = widgets.Button(description="Predict Match Outcome")

    def on_button_click(b):
        home_team = home_team_widget.value
        away_team = away_team_widget.value
        referee = referee_widget.value
        pred_home, pred_away = predict_match(home_team, away_team, referee)
        print(f"Predicted Score: {home_team} {pred_home} - {pred_away} {away_team}")

    button.on_click(on_button_click)

    display(home_team_widget, away_team_widget, referee_widget, button)

except ImportError:
    if __name__ == "__main__":
        print("Football Score Predictor")
        while True:
            home_team = input("Enter Home Team (or type 'exit' to stop): ").strip()
            if home_team.lower() == 'exit':
                print("Exiting program.")
                break

            away_team = input("Enter Away Team: ").strip()
            referee = input("Enter Referee: ").strip()

            if home_team not in df["HomeTeam"].unique() or away_team not in df["AwayTeam"].unique():
                print("Error: One or both teams are not in the dataset. Try again.")
                continue

            pred_home, pred_away = predict_match(home_team, away_team, referee)
            print(f"\nPredicted Score: {home_team} {pred_home} - {pred_away} {away_team}\n")

Text(value='', placeholder='Enter Home Team')

Text(value='', placeholder='Enter Away Team')

Text(value='', placeholder='Enter Referee')

Button(description='Predict Match Outcome', style=ButtonStyle())

Predicted Score: Man United 2 - 1 Chelsea
Predicted Score: Man United 0 - 2 Man City
